# Entity Feature Extraction & Text Cleaning Pipeline
## Two-Tower Business Entity Resolution Preprocessing (Expanded Legal Forms)

This notebook implements the **row-wise and chunk-wise entity feature extraction and normalization pipeline** for business entity matching across datasets (Source 1 $\rightarrow$ Source 2 & Source 3).

### Target Extracted Features
| Field | Two-Tower Usage | Description / Value |
|---|---|---|
| `business_name_raw` | No (Raw Reference) | Raw original name for auditing and reference |
| `name_norm` | **Yes (Primary Name)** | Core native business name (Unicode preserved, lowercase, legal suffixes removed) |
| `name_ascii` | **Yes (Transliteration)** | Accent-free ASCII copy (`bécane` $\leftrightarrow$ `becane`) for cross-source matching |
| `legal_form` | **Yes (Legal Form Text)** | Canonical legal form (e.g. `private_limited`, `llc`, `incorporated`, `pc`, `pllc`, `trust`), or `missing` if none |
| `address_norm` | **Yes (Address Tower)** | Normalized address text (lowercase, punctuation cleaned) for disambiguation |
| `country_norm` | **Yes (Country Filter/Text)** | Standardized lowercase country string for open-set partitioning |

---
### Key Enhancements:
1. **Expanded Legal Form Dictionary (121 Rules)**:
   - **US & English**: `LLC`, `PLLC`, `Inc`, `Corp`, `LLP`, `LP` (Limited Partnership), `PC` (Professional Corporation), `PA` (Professional Association), `Pty Ltd`, `OPC`, `Section 8`, `Co Ltd`.
   - **Truncated Suffixes**: Covers Source 2/Source 3 truncated endings like `Private` or `Pvt` (e.g. `Radiant Marbles Private` $\rightarrow$ `radiant marbles` + `private_limited`).
   - **Trusts & Societies**: `Charitable Trust`, `Memorial Trust`, `Educational Society`, `Welfare Society`, `Seva Samiti`.
   - **Indic Regional Scripts**: Cross-lingual entity matching for Hindi/Devanagari (`प्राइवेट लिमिटेड`, `लिमिटेड`), Tamil (`பிரைவேட் லிமிடெட்`), Telugu (`ప్రైవేట్ లిమిటెడ్`), Kannada (`ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್`), Malayalam (`പ്രൈവറ്റ് ലിമിറ്റഡ്`), Gujarati (`પ્રાઇવેટ લિમિટેડ`), Bengali (`প্রাইভেট লিমিটেড`), Odia, and Punjabi.
2. **Clean Format**: No redundant tags (`[LEGAL]` removed), clean comma-separated canonical names, and `'missing'` when absent.
3. **No Pincode / Fragile Geo Columns**: Unstructured addresses frequently confuse house numbers with postal codes; therefore, pincode column is omitted to prevent noisy features.
4. **Memory-Safe Streaming**: Processes millions of rows in memory-safe chunks of 100,000 rows directly to disk.


In [ ]:
# !pip install unidecode

  Using cached Unidecode-1.4.0-py3-none-any.whl.metadata (13 kB)
Using cached Unidecode-1.4.0-py3-none-any.whl (235 kB)


In [1]:
import os
import sys
import re
import csv
import time
import unicodedata
import unidecode
from typing import Optional, Tuple
import pandas as pd
import numpy as np

# Configure Pandas display for clear readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")


Python version: 3.10.19
Pandas version: 2.3.3


## 1. Expanded Predefined Legal Form Dictionary & Regular Expressions

Below is the expanded dictionary covering:
1. US Corporate forms (`LLC`, `Inc`, `Corp`, `LLP`, `LP`, `PC`, `PA`, `PSC`)
2. Commonwealth forms (`Private Limited`, `Pvt Ltd`, `Pty Ltd`, `Public Limited`, `PLC`)
3. Indian truncated and specialized forms (`Private`, `Pvt`, `OPC`, `Producer Company`)
4. Non-profit / Trust forms (`Charitable Trust`, `Memorial Trust`, `Society`, `Samiti`, `Foundation`)
5. European corporate forms (`GmbH`, `AG`, `SARL`, `SAS`, `SRL`, `BV`, `NV`, `SL`, `Sp z o.o.`)
6. Indic regional script legal forms (`प्राइवेट लिमिटेड`, `பிரைவேட் லிமிடெட்`, `പ്രൈവറ്റ് ലിമിറ്റഡ്`, etc.)


In [2]:
# ============================================================================
# Comprehensive Predefined Legal Forms Mapping Dictionary
# ============================================================================
LEGAL_FORMS = {
    # --- 1. Private Limited Variants (English - India, UK, Commonwealth) ---
    "private limited": "private_limited",
    "pvt limited": "private_limited",
    "private ltd": "private_limited",
    "pvt ltd": "private_limited",
    "p limited": "private_limited",
    "p ltd": "private_limited",
    "pvtltd": "private_limited",
    "prvt ltd": "private_limited",
    "prvt limited": "private_limited",
    "private": "private_limited",  # Truncated ending common in Source 2/3
    "pvt": "private_limited",      # Truncated ending common in Source 2/3
    
    # --- 2. Limited Liability Company (LLC / PLLC - US & Global) ---
    "limited liability company": "llc",
    "limited liability co": "llc",
    "llc": "llc",
    "l l c": "llc",
    "professional limited liability company": "pllc",
    "pllc": "pllc",
    "p l l c": "pllc",
    
    # --- 3. Limited Liability Partnership & Limited Partnership (LLP, LP, LLLP) ---
    "limited liability partnership": "llp",
    "llp": "llp",
    "l l p": "llp",
    "limited liability limited partnership": "lllp",
    "lllp": "lllp",
    "limited partnership": "lp",
    "lp": "lp",
    "l p": "lp",
    
    # --- 4. Professional Corporations & Associations (US Healthcare / Legal) ---
    "professional corporation": "pc",
    "prof corp": "pc",
    "pc": "pc",
    "p c": "pc",
    "professional association": "pa",
    "professional service corporation": "psc",
    "psc": "psc",
    
    # --- 5. Incorporated & Corporation (US, Canada, Global) ---
    "incorporated": "incorporated",
    "inc": "incorporated",
    "corporation": "corporation",
    "corp": "corporation",
    
    # --- 6. Public Limited & Proprietary Limited (UK, Australia, India) ---
    "public limited company": "public_limited",
    "public limited": "public_limited",
    "plc": "public_limited",
    "proprietary limited": "pty_ltd",
    "proprietary ltd": "pty_ltd",
    "pty limited": "pty_ltd",
    "pty ltd": "pty_ltd",
    "pty": "pty_ltd",
    
    # --- 7. Specialized Indian Entities (OPC, Producer Co, Section 8) ---
    "one person company": "opc",
    "opc private limited": "opc",
    "opc pvt ltd": "opc",
    "opc": "opc",
    "producer company limited": "producer_company",
    "producer company": "producer_company",
    "section 8 company": "section_8",
    
    # --- 8. Standard Limited & Company Suffixes ---
    "company limited": "company_limited",
    "co limited": "company_limited",
    "co ltd": "company_limited",
    "and company": "company",
    "and co": "company",
    "ltd": "limited",
    
    # --- 9. Trusts, Societies & Foundations (India / US Non-profits) ---
    "charitable trust": "trust",
    "educational trust": "trust",
    "memorial trust": "trust",
    "welfare trust": "trust",
    "trust": "trust",
    "educational society": "society",
    "welfare society": "society",
    "seva samiti": "society",
    "samiti": "society",
    "society": "society",
    "foundation": "foundation",
    
    # --- 10. European Corporate Forms ---
    "gmbh & co kg": "gmbh_co_kg",
    "gmbh co kg": "gmbh_co_kg",
    "gmbh": "gmbh",
    "ag": "ag",
    "sarl": "sarl",
    "sas": "sas",
    "srl": "srl",
    "bv": "bv",
    "nv": "nv",
    "sl": "sl",
    "sp z oo": "sp_z_oo",
    "spzoo": "sp_z_oo",
    
    # --- 11. Indic Regional Script Legal Forms (Hindi, Tamil, Telugu, etc.) ---
    # Hindi / Marathi (Devanagari)
    "प्राइवेट लिमिटेड": "private_limited",
    "प्रा लि": "private_limited",
    "प्रा. लि.": "private_limited",
    "प्रा.लि.": "private_limited",
    "पब्लिक लिमिटेड": "public_limited",
    "लिमिटेड": "limited",
    "लि.": "limited",
    "लि": "limited",
    "ट्रस्ट": "trust",
    "समिति": "society",
    "सोसायटी": "society",
    "सोसाइटी": "society",
    "फाउंडेशन": "foundation",
    
    # Tamil
    "பிரைவேட் லிமிடெட்": "private_limited",
    "லிமிடெட்": "limited",
    "டிரஸ்ட்": "trust",
    
    # Telugu
    "ప్రైవేట్ లిమిటెడ్": "private_limited",
    "లిమిటెడ్": "limited",
    "ట్రస్ట్": "trust",
    
    # Kannada
    "ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್": "private_limited",
    "ಲಿಮಿಟೆಡ್": "limited",
    "ಟ್ರస్ట్": "trust",
    
    # Malayalam
    "പ്രൈവറ്റ് ലിമിറ്റഡ്": "private_limited",
    "ലിമിറ്റഡ്": "limited",
    "ട്രസ്റ്റ്": "trust",
    
    # Gujarati
    "પ્રાઇવેટ લિમિટેડ": "private_limited",
    "પ્રા લિ": "private_limited",
    "પ્રા. લિ.": "private_limited",
    "લિમિટેડ": "limited",
    "લિ.": "limited",
    "લિ": "limited",
    "ટ્રસ્ટ": "trust",
    
    # Bengali
    "প্রাইভেট লিমিটেড": "private_limited",
    "লিমিটেড": "limited",
    "ট্রাস্ট": "trust",
    
    # Odia
    "ପ୍ରାଇଭେଟ୍ ଲିମିଟେଡ୍": "private_limited",
    "ଲିମିଟେଡ୍": "limited",
    "ଟ୍ରଷ୍ଟ": "trust",
    
    # Punjabi
    "ਪ੍ਰਾਈਵੇਟ ਲਿਮਿਟੇਡ": "private_limited",
    "ਲਿਮਿਟੇਡ": "limited",
    "ਟਰੱਸਟ": "trust",
}

# Sort dictionary keys by length in descending order to match longest phrases first
SORTED_LEGAL_PATTERNS = sorted(LEGAL_FORMS.keys(), key=len, reverse=True)

# Regex matching with boundary awareness for both Latin and Indic scripts
LEGAL_REGEX = re.compile(
    r'(?:(?<=\s)|(?<=^)|(?<=\b))(' + '|'.join(re.escape(term) for term in SORTED_LEGAL_PATTERNS) + r')(?:(?=\s)|(?=$)|(?=\b))',
    re.IGNORECASE
)

# Helper regexes: preserves all Indic scripts (ऀ-ൿ)
APOSTROPHE_REGEX = re.compile(r"['’`]")
PUNCT_CLEAN_REGEX = re.compile(r'[^\w\s\u0900-\u0D7F]', re.UNICODE)
WHITESPACE_REGEX = re.compile(r'\s+')
ASCII_CLEAN_REGEX = re.compile(r'[^a-z0-9\s]')

# Standardized output columns
OUTPUT_COLUMNS = [
    "entity_id",
    "business_name_raw",
    "name_norm",
    "name_ascii",
    "legal_form",
    "address_norm",
    "country_norm",
]

print(f"Total legal form rules defined: {len(LEGAL_FORMS)}")


Total legal form rules defined: 122


## 2. Core Normalization & Feature Extraction Functions

- `normalize_name`: Lowercases, removes apostrophes directly, cleans punctuation, extracts canonical legal forms, strips them from the name to produce `name_norm`, and creates accent-free `name_ascii`.
- `normalize_address`: Lowercases, removes punctuation, collapses whitespace.
- `normalize_country`: Standardizes country string (lowercase, trimmed).
- `extract_entity_features`: Master row-wise function that accepts an entity record and returns a dictionary with the required fields.


In [3]:
def normalize_name(raw_name: Optional[str]) -> Tuple[str, str, str, str]:
    """
    Normalizes a business name:
    1. Removes apostrophes directly (Orelee's -> orelees, McDonald's -> mcdonalds)
    2. Replaces punctuation with spaces while preserving native Unicode and Indic characters.
    3. Searches for and extracts legal forms using the predefined dictionary.
    4. Removes matched legal forms from the name to produce name_norm.
    5. Returns (raw_name, name_norm, name_ascii, legal_form_str).
       - If legal forms are found: returns clean canonical string (e.g. 'private_limited' or 'llc, private_limited')
       - If no legal form is found: returns 'missing'
    """
    if raw_name is None or pd.isna(raw_name):
        return "", "", "", "missing"
    
    raw_str = str(raw_name).strip()
    if not raw_str:
        return "", "", "", "missing"
    
    # Step 1: Lowercase and strip apostrophes directly
    clean_str = APOSTROPHE_REGEX.sub('', raw_str.lower())
    
    # Step 2: Replace remaining punctuation with spaces, preserving Unicode & Indic letters
    clean_str = PUNCT_CLEAN_REGEX.sub(' ', clean_str)
    
    # Step 3: Collapse whitespace
    clean_str = WHITESPACE_REGEX.sub(' ', clean_str).strip()
    
    # Step 4: Extract all legal forms via boundary-aware regex
    matches = LEGAL_REGEX.findall(clean_str)
    if matches:
        seen = set()
        canonical_forms = []
        for m in matches:
            canon = LEGAL_FORMS[m.lower()]
            if canon not in seen:
                seen.add(canon)
                canonical_forms.append(canon)
        legal_form_str = ', '.join(canonical_forms)
    else:
        legal_form_str = 'missing'
            
    # Step 5: Remove matched legal forms from the name string
    name_clean = LEGAL_REGEX.sub(' ', clean_str)
    name_norm = WHITESPACE_REGEX.sub(' ', name_clean).strip()
    
    # Safety fallback: if stripping legal forms emptied the name completely (e.g. "Memorial Trust"),
    # fall back to clean_str so name_norm is never empty or NaN
    if not name_norm:
        name_norm = clean_str
    
    # Step 6: Create transliterated ASCII version (name_ascii)
    try:
        raw_ascii = unidecode.unidecode(name_norm).lower()
        raw_ascii = ASCII_CLEAN_REGEX.sub(' ', raw_ascii)
        name_ascii = WHITESPACE_REGEX.sub(' ', raw_ascii).strip()
        if not name_ascii:
            name_ascii = name_norm
    except Exception:
        name_ascii = unicodedata.normalize('NFKD', name_norm).encode('ascii', 'ignore').decode('utf-8')
        name_ascii = WHITESPACE_REGEX.sub(' ', name_ascii).strip()
        if not name_ascii:
            name_ascii = name_norm
    
    return raw_str, name_norm, name_ascii, legal_form_str


def normalize_address(raw_address: Optional[str]) -> str:
    """
    Normalizes a business address:
    - Lowercase
    - Strip apostrophes
    - Replace punctuation with spaces
    - Collapse extra whitespace
    """
    if raw_address is None or pd.isna(raw_address):
        return ""
    addr = str(raw_address).strip().lower()
    addr = APOSTROPHE_REGEX.sub('', addr)
    addr = PUNCT_CLEAN_REGEX.sub(' ', addr)
    return WHITESPACE_REGEX.sub(' ', addr).strip()


def normalize_country(raw_country: Optional[str]) -> str:
    """
    Normalizes country string: lowercase and trimmed.
    Supports open-set countries (US, India, France, Germany, etc.).
    """
    if raw_country is None or pd.isna(raw_country):
        return ""
    return str(raw_country).strip().lower()


def extract_entity_features(row: dict) -> dict:
    """
    Master Row-Wise Extraction Function:
    Accepts a dictionary (or Pandas Series / tuple) representing a single entity row
    and extracts all normalized features required for the Two-Tower model.
    """
    entity_id = row.get('entity_id', '') if isinstance(row, dict) else row['entity_id']
    raw_name = row.get('business_name', '') if isinstance(row, dict) else row['business_name']
    raw_addr = row.get('business_address', '') if isinstance(row, dict) else row['business_address']
    raw_country = row.get('country', '') if isinstance(row, dict) else row['country']
    
    name_raw, name_norm, name_ascii, legal_form = normalize_name(raw_name)
    addr_norm = normalize_address(raw_addr)
    country_norm = normalize_country(raw_country)
    
    return {
        "entity_id": str(entity_id),
        "business_name_raw": name_raw,
        "name_norm": name_norm,
        "name_ascii": name_ascii,
        "legal_form": legal_form,
        "address_norm": addr_norm,
        "country_norm": country_norm,
    }

print("Normalization functions successfully defined.")


Normalization functions successfully defined.


## 3. Unit Verification on Target Test Cases (English, International & Indic Scripts)

Let us test the normalization function across a wide spectrum of cases:
1. Standard English forms (`LLC`, `Pvt Ltd`, `Inc`)
2. Professional forms (`PC`, `PLLC`, `LP`)
3. Trusts & Societies (`Celestial Memorial Trust`, `Kolkata Educational Society`)
4. Truncated Indian forms (`Radiant Marbles Private`)
5. Indic Regional Scripts (Hindi, Malayalam, Tamil)


In [4]:
benchmark_cases = [
    "LLC Bécane Coffee Pvt. Ltd.",
    "Becane LLC Coffee",
    "Becane Coffee Pvt Ltd",
    "B+ Retail Inc",
    "Orelee's Barbershop",
    "Consulting Nyasa Nursing Private Limited",
    "Babb S. Johnson, DO, DDS PC",
    "Radiology Consultants PLLC",
    "Williamson Quality Braves LP",
    "Celestial Memorial Trust",
    "Kolkata Educational Society",
    "Radiant Marbles Private",
    "रिलायंस इंडस्ट्रीज लिमिटेड",
    "അൽ കൺസ്ട്രക്ഷൻസ് ഫുഡ്സ് പ്രൈവറ്റ് ലിമിറ്റഡ്",
]

test_results = []
for name in benchmark_cases:
    raw, n_norm, n_ascii, l_form = normalize_name(name)
    test_results.append({
        "Input Raw Name": raw,
        "name_norm": n_norm,
        "name_ascii": n_ascii,
        "legal_form": l_form,
    })

test_df = pd.DataFrame(test_results)
test_df


,Input Raw Name,name_norm,name_ascii,legal_form
0,LLC Bécane Coffee Pvt. Ltd.,bécane coffee,becane coffee,"llc, private_limited"
1,Becane LLC Coffee,becane coffee,becane coffee,llc
2,Becane Coffee Pvt Ltd,becane coffee,becane coffee,private_limited
3,B+ Retail Inc,b retail,b retail,incorporated
4,Orelee's Barbershop,orelees barbershop,orelees barbershop,missing
5,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing,consulting nyasa nursing,private_limited
6,"Babb S. Johnson, DO, DDS PC",babb s johnson do dds,babb s johnson do dds,pc
7,Radiology Consultants PLLC,radiology consultants,radiology consultants,pllc
8,Williamson Quality Braves LP,williamson quality braves,williamson quality braves,lp
9,Celestial Memorial Trust,celestial,celestial,trust


## 4. Processing 10 Rows from `dataset/train/train_source1.tsv`

We verify row-by-row on the first 10 rows of the dataset.


In [5]:
DATASET_PATH = os.path.join("..", "..", "dataset", "train", "train_source1.tsv")
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = "dataset/train/train_source1.tsv"

print(f"Loading 10 sample rows from: {DATASET_PATH}")
df_raw = pd.read_csv(
    DATASET_PATH,
    sep='\t',
    quoting=csv.QUOTE_NONE,
    dtype=str,
    nrows=10
)

# Apply row-wise extraction function
processed_rows = [extract_entity_features(row) for _, row in df_raw.iterrows()]
df_clean = pd.DataFrame(processed_rows)

print(f"Successfully processed {len(df_clean)} sample rows.\n")
df_clean


Loading 10 sample rows from: ..\..\dataset\train\train_source1.tsv
Successfully processed 10 sample rows.



,entity_id,business_name_raw,name_norm,name_ascii,legal_form,address_norm,country_norm
0,S1-925783039,Orelee's Barbershop,orelees barbershop,orelees barbershop,missing,1795 westchester drive high point nc,us
1,S1-773889195,Prime Money,prime money,prime money,missing,17560 ellis road tahlequah ok,us
2,S1-377745466,B+ Retail Inc,b retail,b retail,incorporated,1712 montebello avenue phoenix az,us
3,S1-133037285,Christ Chapel,christ chapel,christ chapel,missing,2100 cameron drive unit apartment g dundalk md,us
4,S1-755362802,Prabhav Business Center,prabhav business center,prabhav business center,missing,797 lake town block a kolkata howrah west bengal,india
5,S1-851869949,Custom Wealth Services LLC,custom wealth services,custom wealth services,llc,oh columbus 5559 orville avenue,us
6,S1-785847572,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing,consulting nyasa nursing,private_limited,2505 tower 1 oakwood runwal greens mulund goreagon link road near fortis hospital bhandup west mumbai maharashtra,india
7,S1-27541239,Nexus Anchor Rain,nexus anchor rain,nexus anchor rain,missing,1111 church street unit 2007 nashville tn,us
8,S1-629417405,Moore Bitwise Inc,moore bitwise,moore bitwise,incorporated,337 oakland avenue michigan city in,us
9,S1-22305073,Dermatology Green Medicine,dermatology green medicine,dermatology green medicine,missing,294 meadowcreek drive unit unit 2 village of pewaukee wi,us


## 5. Chunk-Wise Processing Function for Large Datasets

Now we define `preprocess_tsv_chunkwise(input_path, output_path, chunksize=100000)`.
This function processes any dataset in memory-safe batches and streams clean records to disk.


In [6]:
def preprocess_tsv_chunkwise(
    input_path: str,
    output_path: str,
    chunksize: int = 100_000,
    max_rows: Optional[int] = None,
    verbose: bool = True
) -> dict:
    """
    Reads an input TSV file in chunks, processes every entity row-wise,
    and appends the cleaned records to output_path.
    """
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Input file not found: {input_path}")

    out_dir = os.path.dirname(output_path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)

    if verbose:
        print("=" * 70)
        print(" CHUNK-WISE ENTITY PREPROCESSING PIPELINE (EXPANDED DICTIONARY)")
        print("=" * 70)
        print(f"Input File:   {input_path}")
        print(f"Output File:  {output_path}")
        print(f"Chunk Size:   {chunksize:,} rows")
        print(f"Legal Rules:  {len(LEGAL_FORMS)} defined patterns")
        if max_rows:
            print(f"Max Rows:     {max_rows:,} rows (TEST MODE)")
        print("-" * 70)

    start_time = time.time()
    total_processed = 0
    chunk_index = 0
    first_chunk = True

    reader = pd.read_csv(
        input_path,
        sep='\t',
        quoting=csv.QUOTE_NONE,
        dtype=str,
        chunksize=chunksize
    )

    for chunk_df in reader:
        chunk_start = time.time()
        chunk_index += 1

        if max_rows is not None and total_processed + len(chunk_df) > max_rows:
            chunk_df = chunk_df.iloc[:max_rows - total_processed]

        cleaned_records = []
        for row in chunk_df.itertuples(index=False):
            eid = str(row[0]) if pd.notna(row[0]) else ""
            bname = str(row[1]) if pd.notna(row[1]) else ""
            baddr = str(row[2]) if pd.notna(row[2]) else ""
            ctry = str(row[3]) if pd.notna(row[3]) else ""

            raw_name, n_norm, n_ascii, l_form = normalize_name(bname)
            addr_norm = normalize_address(baddr)
            c_norm = normalize_country(ctry)

            cleaned_records.append((
                eid,
                raw_name,
                n_norm,
                n_ascii,
                l_form,
                addr_norm,
                c_norm
            ))

        out_chunk_df = pd.DataFrame(cleaned_records, columns=OUTPUT_COLUMNS)
        out_chunk_df.to_csv(
            output_path,
            sep='\t',
            index=False,
            mode='w' if first_chunk else 'a',
            header=first_chunk,
            quoting=csv.QUOTE_MINIMAL
        )
        first_chunk = False
        total_processed += len(cleaned_records)

        chunk_elapsed = time.time() - chunk_start
        cumulative_elapsed = time.time() - start_time
        overall_rate = total_processed / cumulative_elapsed if cumulative_elapsed > 0 else 0

        if verbose:
            print(
                f"[Chunk {chunk_index:3d}] Processed {len(cleaned_records):,d} rows "
                f"in {chunk_elapsed:.2f}s | "
                f"Total: {total_processed:,d} rows "
                f"({cumulative_elapsed:.1f}s, {overall_rate:,.0f} rows/s)"
            )

        if max_rows is not None and total_processed >= max_rows:
            break

    total_time = time.time() - start_time
    file_size_mb = os.path.getsize(output_path) / (1024 * 1024) if os.path.exists(output_path) else 0

    if verbose:
        print("=" * 70)
        print(" PREPROCESSING COMPLETE")
        print("=" * 70)
        print(f"Total Rows Processed: {total_processed:,}")
        print(f"Total Time Elapsed:   {total_time:.2f} seconds ({total_time / 60:.2f} minutes)")
        print(f"Average Throughput:   {total_processed / total_time:,.0f} rows/second")
        print(f"Output File Size:     {file_size_mb:.2f} MB")
        print(f"Output Location:      {output_path}")
        print("=" * 70)

    return {
        "input_path": input_path,
        "output_path": output_path,
        "total_rows": total_processed,
        "total_seconds": total_time,
        "rows_per_second": total_processed / total_time if total_time > 0 else 0,
        "file_size_mb": file_size_mb,
    }

print("preprocess_tsv_chunkwise function defined.")


preprocess_tsv_chunkwise function defined.


## 6. Execution: Preprocessing Full `train_source1.tsv` Chunk-Wise

We execute `preprocess_tsv_chunkwise` on `dataset/train/train_source1.tsv` saving to:
`data/pre-processing/preprocessed_train_sounce1.tsv`


In [9]:
INPUT_FILE = os.path.join("..", "..", "dataset", "test", "test_source3.tsv")
if not os.path.exists(INPUT_FILE):
    INPUT_FILE = "dataset/test/test_source3.tsv"

OUTPUT_FILE = os.path.join("..", "..", "data", "final_test_data", "preprocessed_test_source3.tsv")
if not os.path.exists(os.path.dirname(OUTPUT_FILE)):
    OUTPUT_FILE = "data/final_test_data/preprocessed_test_source3.tsv"

# Run chunk-wise preprocessing for the complete dataset (2.2M rows)
metrics = preprocess_tsv_chunkwise(
    input_path=INPUT_FILE,
    output_path=OUTPUT_FILE,
    chunksize=100_000,
    verbose=True
)


 CHUNK-WISE ENTITY PREPROCESSING PIPELINE (EXPANDED DICTIONARY)
Input File:   ..\..\dataset\test\test_source3.tsv
Output File:  ..\..\data\final_test_data\preprocessed_test_source3.tsv
Chunk Size:   100,000 rows
Legal Rules:  122 defined patterns
----------------------------------------------------------------------
[Chunk   1] Processed 100,000 rows in 3.25s | Total: 100,000 rows (3.4s, 29,185 rows/s)
[Chunk   2] Processed 100,000 rows in 3.37s | Total: 200,000 rows (7.0s, 28,663 rows/s)
[Chunk   3] Processed 100,000 rows in 3.32s | Total: 300,000 rows (10.5s, 28,629 rows/s)
[Chunk   4] Processed 100,000 rows in 3.28s | Total: 400,000 rows (13.9s, 28,730 rows/s)
[Chunk   5] Processed 100,000 rows in 3.28s | Total: 500,000 rows (17.4s, 28,778 rows/s)
[Chunk   6] Processed 100,000 rows in 3.40s | Total: 600,000 rows (20.9s, 28,646 rows/s)
[Chunk   7] Processed 100,000 rows in 3.36s | Total: 700,000 rows (24.5s, 28,598 rows/s)
[Chunk   8] Processed 100,000 rows in 3.33s | Total: 800,000 

## 7. Verification of Output File

Let us verify the generated TSV file by reading back the header and first 10 rows.


In [8]:
df_saved = pd.read_csv(OUTPUT_FILE, sep='\t', nrows=10)
print(f"File verified! Saved shape preview: {df_saved.shape}")
df_saved


File verified! Saved shape preview: (10, 7)


,entity_id,business_name_raw,name_norm,name_ascii,legal_form,address_norm,country_norm
0,S1-925783039,Orelee's Barbershop,orelees barbershop,orelees barbershop,missing,1795 westchester drive high point nc,us
1,S1-773889195,Prime Money,prime money,prime money,missing,17560 ellis road tahlequah ok,us
2,S1-377745466,B+ Retail Inc,b retail,b retail,incorporated,1712 montebello avenue phoenix az,us
3,S1-133037285,Christ Chapel,christ chapel,christ chapel,missing,2100 cameron drive unit apartment g dundalk md,us
4,S1-755362802,Prabhav Business Center,prabhav business center,prabhav business center,missing,797 lake town block a kolkata howrah west bengal,india
5,S1-851869949,Custom Wealth Services LLC,custom wealth services,custom wealth services,llc,oh columbus 5559 orville avenue,us
6,S1-785847572,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing,consulting nyasa nursing,private_limited,2505 tower 1 oakwood runwal greens mulund goreagon link road near fortis hospital bhandup west mumbai maharashtra,india
7,S1-27541239,Nexus Anchor Rain,nexus anchor rain,nexus anchor rain,missing,1111 church street unit 2007 nashville tn,us
8,S1-629417405,Moore Bitwise Inc,moore bitwise,moore bitwise,incorporated,337 oakland avenue michigan city in,us
9,S1-22305073,Dermatology Green Medicine,dermatology green medicine,dermatology green medicine,missing,294 meadowcreek drive unit unit 2 village of pewaukee wi,us


## 8. Simple Function to Process Any Dataset (`clean_dataset`)

Call `clean_dataset('<name>')` with any dataset:
- `'train_source2'`, `'train_source3'`
- `'test_source1'`, `'test_source2'`, `'test_source3'`


In [14]:
def clean_dataset(name: str, max_rows: int = None, chunksize: int = 100_000):
    """Clean any dataset chunk-wise and return a 5-row preview DataFrame."""
    base_name = os.path.splitext(os.path.basename(name))[0]
    split = 'test' if 'test' in base_name else 'train'
    
    # 1. Input path
    in_path = name if os.path.exists(name) else os.path.join('..', '..', 'dataset', split, f'{base_name}.tsv')
    if not os.path.exists(in_path):
        in_path = os.path.join('dataset', split, f'{base_name}.tsv')
        
    # 2. Output path
    out_path = os.path.join('..', '..', 'data', 'pre-processing', f'preprocessed_{base_name}.tsv')
    if not os.path.exists(os.path.dirname(out_path)):
        out_path = os.path.join('data', 'pre-processing', f'preprocessed_{base_name}.tsv')
        
    # 3. Process chunk-wise
    preprocess_tsv_chunkwise(in_path, out_path, chunksize=chunksize, max_rows=max_rows)
    
    # 4. Return preview DataFrame
    return pd.read_csv(out_path, sep='\t', nrows=5)


### Example Usage:
Test with 10 rows on `train_source2` and `test_source2`:


In [15]:
# Test on train_source2 (10 rows)
clean_dataset('train_source2', max_rows=10)


NameError: name 'preprocess_tsv_chunkwise' is not defined

In [ ]:
# Test on test_source2 (10 rows - includes France / European entities)
clean_dataset('test_source2', max_rows=10)


In [ ]:
# To process full datasets (millions of rows), simply run:
# clean_dataset('train_source2')
# clean_dataset('train_source3')
# clean_dataset('test_source1')
# clean_dataset('test_source2')
# clean_dataset('test_source3')
